# ⚡ OmniVoice AI Studio - 1-Click Google Colab Launcher
Nền tảng lồng tiếng & Clone Voice kịch bản chuyên nghiệp cho **600+ ngôn ngữ**.

### 🌟 Tính năng tối ưu đặc biệt cho Google Colab:
- 💾 **Lưu trữ vĩnh viễn trên Google Drive**: Toàn bộ Voice Profile `.pt`, Cache model HuggingFace & Torch, Audio kết quả không bị mất khi Colab khởi động lại.
- ⚡ **Đường truyền siêu tốc Cloudflare Quick Tunnel**: Ổn định gấp 5-10x so với link Gradio mặc định, tải audio kịch bản mượt mà không bị timeout.
- 📊 **Giám sát thời gian thực (Real-time ETA & VRAM Monitor)**: Theo dõi tiến độ chính xác từng phân đoạn khi sinh kịch bản dài.
- 🛡️ **Anti-Disconnect / Keep-Alive**: Tránh bị ngắt kết nối Colab khi treo máy sinh batch.

### 1️⃣ Bước 1: Kết nối Google Drive & Kiểm tra GPU

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Kiểm tra GPU
!nvidia-smi

### 2️⃣ Bước 2: Clone source code & Cài đặt thư viện siêu tốc bằng `uv`

In [ ]:
import os

%cd /content
# Clone repo nếu chưa có, hoặc cập nhật nếu đã có
if not os.path.exists('/content/OmniVoice'):
    !git clone https://github.com/k2-fsa/OmniVoice.git /content/OmniVoice

%cd /content/OmniVoice

# Cài đặt uv package manager để tăng tốc cài thư viện x10 lần
!pip install -q uv
!uv pip install --system -e . gradio soundfile google-genai

# Cài đặt Cloudflared binary cho Linux nếu chưa có
!wget -q -nc -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared
!cp /tmp/cloudflared /usr/local/bin/cloudflared

### 3️⃣ Bước 3: Anti-Disconnect / Keep-Alive Helper (Tùy chọn cho Colab Browser)

In [ ]:
from IPython.display import display, HTML

# Keep-Alive script chạy trực tiếp trên tab Colab
display(HTML('''
<script>
function ClickConnect(){
  console.log("⚡ Colab Keep-Alive Heartbeat Triggered"); 
  document.querySelector("colab-connect-button")?.click() 
}
setInterval(ClickConnect, 60000);
</script>
<div style="color: #10b981; font-weight: bold;">✅ Anti-Disconnect Keep-Alive đã kích hoạt!</div>
'''))

### 4️⃣ Bước 4: Khởi chạy OmniVoice AI Studio với Cloudflare Tunnel
*(Đường link public tốc độ cao `https://xxxx.trycloudflare.com` sẽ xuất hiện ngay sau khi khởi động)*

In [ ]:
import os
%cd /content/OmniVoice

# Cấu hình Google Drive Cache vĩnh viễn
os.environ["HF_HOME"] = "/content/drive/MyDrive/OmniVoice_Studio/hf_cache"
os.environ["TORCH_HOME"] = "/content/drive/MyDrive/OmniVoice_Studio/torch_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/content/drive/MyDrive/OmniVoice_Studio/hf_cache"

# Khởi chạy giao diện với Cloudflare Tunnel
!python -m omnivoice.cli.demo --model k2-fsa/OmniVoice --tunnel cloudflare